In [1]:
import pandas as pd
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

In [3]:
def get_list_of_files_in_s3_location(cls_client, str_bucket_name, str_prefix):
    dict_response = cls_client.list_objects_v2(
        Bucket=str_bucket_name,
        Prefix=str_prefix,
    )
    list_dict_contents = dict_response['Contents']
    list_str_file = [dict_contents['Key'] for dict_contents in list_dict_contents]
    list_str_filename = [str_file.split('/')[-1] for str_file in list_str_file]
    list_str_filename = [str_filename for str_filename in list_str_filename if 'gzip' in str_filename]
    return list_str_filename

### Constants

In [4]:
str_image_name = 'simple-model-test-parse-tbldove'
int_iteration = 1
str_instance = 'm5.2xlarge'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

int_vcpu: 8
int_memory_gb: 32
int_memory_mebibytes: 30518


### Get number of jobs

In [5]:
cls_client = boto3.client('s3')
list_str_filename = get_list_of_files_in_s3_location(
    cls_client=cls_client,
    str_bucket_name='20241022-parse-snowflake-payloads',
    str_prefix='deprecated/03_pull_payloads_tbldove',
)
int_n_jobs = len(list_str_filename)
print(f'Number of Files: {int_n_jobs}')

Number of Files: 855


### Save to s3

In [6]:
df = pd.DataFrame({'str_filename': list_str_filename})
str_filename = 'df_list_str_filename.csv'
str_uri = f's3://20241112-simple-model-test/filenames_for_parsing/{str_filename}'
df.to_csv(str_uri, index=False)
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,str_filename
0,df_requests_2021-07-26.gzip
1,df_requests_2021-07-27.gzip
2,df_requests_2021-07-28.gzip
3,df_requests_2021-07-29.gzip
4,df_requests_2021-07-30.gzip
...,...
850,df_requests_2023-11-23.gzip
851,df_requests_2023-11-24.gzip
852,df_requests_2023-11-25.gzip
853,df_requests_2023-11-26.gzip


### Create compute environment

In [7]:
# initialize class
cls_client = boto3.client('batch')

In [8]:
# get role
try:
    str_role = get_execution_role()
except:
    ! pip install --upgrade boto3
    str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '187',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:11:19 GMT',
                                      'x-amz-apigw-id': 'BQC1sGVjvHcEPIg=',
                                      'x-amzn-requestid': '36f8f764-fecf-40e2-9c26-528eacae5b05',
                                      'x-amzn-trace-id': 'Root=1-67364b57-6586f0302316c6823d6c7361'},
                      'HTTPStatusCode': 200,
                      'RequestId': '36f8f764-fecf-40e2-9c26-528eacae5b05',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create Job Queue

In [10]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '161',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:11:42 GMT',
                                      'x-amz-apigw-id': 'BQC5YEQRvHcEPMw=',
                                      'x-amzn-requestid': '5825311d-5691-4617-8c7e-0b3a65655dac',
                                      'x-amzn-trace-id': 'Root=1-67364b6e-20344dae51dfaedf4609d39b'},
                      'HTTPStatusCode': 200,
                      'RequestId': '5825311d-5691-4617-8c7e-0b3a65655dac',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [11]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '195',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:11:43 GMT',
                                      'x-amz-apigw-id': 'BQC5ZGNDPHcEayA=',
                                      'x-amzn-requestid': '2dc503f4-466d-4dff-b8ad-14295962043e',
                                      'x-amzn-trace-id': 'Root=1-67364b6e-1403f2394e54cb620c311eec'},
                      'HTTPStatusCode': 200,
                      'RequestId': '2dc503f4-466d-4dff-b8ad-14295962043e',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [14]:
# submit a job (only for testing)
while True:
    try:
        str_job_name = f'job-name-{str_image_name}-{int_iteration}'
        response = cls_client.submit_job(
            jobDefinition=str_job_definition,
            jobQueue=str_job_queue_name,
            jobName=str_job_name,
            arrayProperties={
                'size': int_n_jobs,
            },
        )
        pprint(response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '192',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 14 Nov 2024 19:21:04 GMT',
                                      'x-amz-apigw-id': 'BQERHEZ5PHcEWBA=',
                                      'x-amzn-requestid': '3279df78-4faf-4712-86d0-4a0c3a7f1559',
                                      'x-amzn-trace-id': 'Root=1-67364da0-5a914aa709eecc881f08c89e'},
                      'HTTPStatusCode': 200,
                      'RequestId': '3279df78-4faf-4712-86d0-4a0c3a7f1559',
                      'RetryAttempts': 0},
 'jobArn': 'arn:aws:batch:us-west-2:83669

### Show arns

In [13]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-simple-model-test-parse-tbldove-1
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-simple-model-test-parse-tbldove-1:1
